# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore and process the FAIR² dataset.

### Dataset Source
The dataset is described via a Croissant schema, accessible at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# If running in a new environment, install mlcroissant first
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata object (do not subscript or iterate, print directly)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate all record sets and examine their IDs and available fields.

*Note*: We access and display IDs (`@id`) for all `RecordSet` and their fields.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets declared via dataset.record_sets; trying dataset.list_record_sets() for dynamic discovery...")
    # Try to dynamically discover record sets
    record_sets_ids = dataset.list_record_sets()
else:
    record_sets_ids = [rs['@id'] for rs in record_sets]

print('Record sets found:')
for rs_id in record_sets_ids:
    print(f"- {rs_id}")

# For each record set, print available fields and their @id
print("\nFields for each record set:")
for rs_id in record_sets_ids:
    fields = dataset.list_fields(rs_id)
    print(f"\nRecordSet: {rs_id}")
    for field in fields:
        print(f"  Field: {field['@id']} (name: {field.get('name', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Choose a record set `@id` based on the overview above, and extract its data.

In [ ]:
# Choose one or multiple record sets from above for extraction. Here, we use the first one (replace if needed):
if not record_sets_ids:
    raise ValueError("No record sets found in the dataset.")
selected_record_set_ids = record_sets_ids[:1]  # Adjust if needed to select more

dataframes = {}
for rs_id in selected_record_set_ids:
    print(f"\nLoading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Fields (@id) in this record set:")
    print(df.columns.tolist())
    print(f"First 5 records:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Filter and transform the data using field `@id`s. We'll:
- Select a numeric field by its `@id`
- Filter records, normalize, and group by another field
- All manipulations use the unique `@id` from the earlier overview

In [ ]:
# Example: Filter and normalize a numeric field, then group by a categorical field
import numpy as np

# Pick the record set just loaded
record_set_id = selected_record_set_ids[0]
df = dataframes[record_set_id]

# Automatically pick first numeric field by checking dtypes/common names, or set manually if required:
numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or df[col].dtype in (np.int64, np.float64)]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    raise ValueError("No obvious numeric field found. Please specify manually using its @id.")

# Set example threshold
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a categorical/group field using @id (e.g. 'sex' or 'anatomical_site' if present):
possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'site' in col.lower() or 'group' in col.lower()]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    print(f"Grouping by categorical field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"Grouped data by {group_field_id} (showing group means):")
    display(grouped_df.head())
else:
    print("No categorical/group field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its relationship with the chosen group field (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If group_field_id is defined, boxplot per group
if 'group_field_id' in locals():
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've:
 - Loaded robust clinical data about second primary colorectal cancers from the FAIR² dataset via its Croissant schema
 - Programmatically discovered record sets and fields using unique `@id`s for reproducibility
 - Demonstrated data extraction, filtering, normalization, grouping, and visual analytics
 
You can extend these analyses by exploring additional record sets, adding modeling steps, or joining with external data.

> **Note**: Always reference fields, record sets, and entities by their `@id` as defined by the Croissant schema for consistency and code maintainability.